CONFIGURATION, INGESTION DES DONNEÉS ET COLLECTE DES DONNEÉS SOCIALES (COMMENTAIRE) 

In [4]:
import fastf1
import pandas as pd

def get_performance_data(year, race_name):
    # Chargement de la session de course
    session = fastf1.get_session(year, race_name, 'R')
    session.load(laps=False, telemetry=False, weather=False)
    
    # Sélection des colonnes clés pour l'analyse métier
    results = session.results[['FullName', 'TeamName', 'ClassifiedPosition', 'GridPosition', 'Points', 'Status']]
    
    # Nettoyage : conversion des positions en numérique (0 pour les abandons/DNF)
    results['ClassifiedPosition'] = pd.to_numeric(results['ClassifiedPosition'], errors='coerce').fillna(0)
    return results

def get_social_mock_data():
    # Simulation de données issues de réseaux sociaux pour la phase de test
    data = {
        'timestamp': pd.to_datetime(['2024-05-26'] * 5),
        'author': ['fan_44', 'cl16_daily', 'f1_expert', 'redbull_news', 'mercedes_f1_fan'],
        'text': [
            "What a masterclass from Leclerc in Monaco! Perfect race.",
            "I'm so happy for Charles, he finally won his home GP!",
            "Strategy at Ferrari was actually solid today. Surprising.",
            "Tough weekend for Max, the car didn't look comfortable at all.",
            "Monaco is still a bit boring, no overtakes possible."
        ]
    }
    return pd.DataFrame(data)


df_race = get_performance_data(2024, 'Monaco')
df_social = get_social_mock_data()


df_race.to_csv('race_results.csv', index=False)
df_social.to_csv('social_comments.csv', index=False)

core           INFO 	Loading data for Monaco Grand Prix - Race [v3.8.2]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['16', '81', '55', '4', '63', '1', '44', '22', '23', '10', '14', '3', '77', '18', '2', '24', '31', '11', '27', '20']
/tmp/ipykernel_3956366/1475814283.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  results['ClassifiedPosition'] = pd.to_numeric(results['ClassifiedPosition'], errors='coerce').fillna(0)


In [9]:
from transformers import pipeline


# utilisation d'un modèle spécialisé pour l'anglais
classifier = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english")

def analyze_sentiments(df):
   
    results = df['text'].apply(lambda x: classifier(x)[0])
    
    # On extrait les colonnes Sentiment (Texte) et Score (Numérique)
    df['sentiment_label'] = results.apply(lambda x: x['label'])
    df['sentiment_score'] = results.apply(lambda x: x['score'] if x['label'] == 'POSITIVE' else -x['score'])
    
    return df


df_social_scored = analyze_sentiments(df_social)


df_social_scored.to_csv('f1_sentiment_processed.csv', index=False)

df_social_scored[['text', 'sentiment_label', 'sentiment_score']].head()

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

,text,sentiment_label,sentiment_score
0,What a masterclass from Leclerc in Monaco! Per...,POSITIVE,0.999869
1,"I'm so happy for Charles, he finally won his h...",POSITIVE,0.999866
2,Strategy at Ferrari was actually solid today. ...,POSITIVE,0.999869
3,"Tough weekend for Max, the car didn't look com...",NEGATIVE,-0.999753
4,"Monaco is still a bit boring, no overtakes pos...",NEGATIVE,-0.995495


IMPORTATION DE L'API NEWSAPI pour simuler avec des données réelles isse de magazines et articles de presse

In [1]:
from newsapi import NewsApiClient
import pandas as pd


newsapi = NewsApiClient(api_key='db9d1486459b4d8ab0dcef78119bd7c4')

# On cherche les articles qui mentionnent "Ferrari F1" ou "Leclerc"
data = newsapi.get_everything(q='Ferrari F1 OR Leclerc',
                                  language='en',
                                  sort_by='relevancy',
                                  page_size=100)

articles = data['articles']
df_news = pd.DataFrame(articles)

# le titre et la description pour l'IA
df_news = df_news[['publishedAt', 'title', 'description', 'source']]
df_news.head()

,publishedAt,title,description,source
0,2026-03-19T19:14:07Z,Aston Martin make approach for Audi boss Wheatley,Audi team principal Jonathan Wheatley is wante...,"{'id': None, 'name': 'BBC News'}"
1,2026-04-08T16:29:49Z,"The good, the bad and the ugly of F1's new 202...",F1's new-for-2026 regulations have come in for...,"{'id': 'espn', 'name': 'ESPN'}"
2,2026-04-16T19:29:29Z,He Thought He Was Renting a Ferrari 812—Then P...,It sounds like the kind of story you’d expect ...,"{'id': None, 'name': 'Yahoo Entertainment'}"
3,2026-04-09T15:35:00Z,A Man Was Renting a Ferrari 812 Superfast for ...,The renter was paying over $2000 a month for t...,"{'id': None, 'name': 'RoadandTrack.com'}"
4,2026-03-23T23:10:30Z,The Hisense U7SG is a great midrange TV you sh...,"With all of the focus on RGB LED technology, i...","{'id': 'the-verge', 'name': 'The Verge'}"


UTILISATION DU MODELE distilBERT POUR ANALYSER LES SENTIMENTS EMIS DANS CHAQUE ARTICLE ET POUVOIR LES SCORER

In [4]:
from transformers import pipeline

# Initialisation du modèle

sentiment_pipeline = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english")

# (Titre + Description) 

df_news['full_text'] = df_news['title'].fillna('') + " " + df_news['description'].fillna('')

#  Fonction pour analyser le sentiment (on limite à 512 caractères pour BERT)
def get_sentiment(text):
    if not text.strip(): return "NEUTRAL", 0.0
    # On tronque pour éviter les erreurs de longueur du modèle
    result = sentiment_pipeline(text[:512])[0]
    return result['label'], result['score']


print("Analyse du sentiment en cours...")
df_news[['label', 'score']] = df_news['full_text'].apply(lambda x: pd.Series(get_sentiment(x)))

# Transformation pour Power BI (Score positif/négatif)

df_news['final_score'] = df_news.apply(lambda x: x['score'] if x['label'] == 'POSITIVE' else -x['score'], axis=1)

print("Analyse terminée !")
df_news[['title', 'label', 'final_score']].head()

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Analyse du sentiment en cours...
Analyse terminée !


,title,label,final_score
0,Aston Martin make approach for Audi boss Wheatley,POSITIVE,0.520866
1,"The good, the bad and the ugly of F1's new 202...",POSITIVE,0.661595
2,He Thought He Was Renting a Ferrari 812—Then P...,NEGATIVE,-0.999444
3,A Man Was Renting a Ferrari 812 Superfast for ...,NEGATIVE,-0.998137
4,The Hisense U7SG is a great midrange TV you sh...,POSITIVE,0.989951


LIER LES NEWS AUX PILOTES POUR UNE MEILLEURE VISUALISATION DANS POWER BI

In [5]:
drivers = ['Leclerc', 'Sainz', 'Verstappen', 'Hamilton', 'Norris', 'Alonso']

def tag_driver(text):
    for driver in drivers:
        if driver.lower() in text.lower():
            return driver
    return "General / Other"

df_news['Driver'] = df_news['full_text'].apply(tag_driver)

EXPORTATION DE LA DATA SOUS FORME DE FICHIER CSV POUR LA VISUALISATION

In [6]:
df_news.to_csv('f1_news_sentiment.csv', index=False)